# 11 — Agentrammeverk: LangGraph og CrewAI

**Fase:** 2 — Kjerne AI | **Tid:** 2 timer | **Krav:** Notatbok 10

**Hva du bygger:** Samme pensjonsagent i to ulike rammeverk — slik at du forstår hva de abstraherer og når du bør bruke dem.

---

## Hvorfor rammeverk?

Agenten i notatbok 10 fungerer, men manuell ReAct-løkke blir raskt rotete med:
- Feilhåndtering og retries
- Parallelle verktøykall
- Betingede steg (hvis X → gjør Y, ellers Z)
- Logging og observability

**Rammeverk løser dette med struktur.**

| Rammeverk | Styrke | Passer for |
|-----------|--------|----------|
| **LangGraph** | Tilstandsmaskiner, kompleks flyt | Kontrollerte pipelines |
| **CrewAI** | Rollebaserte multi-agenter | Team av spesialiserte agenter |
| Pydantic AI | Type-sikker, lett | Enkle produksjonsagenter |

---

## Del 1: LangGraph — Agent som tilstandsmaskin

In [ ]:
%pip install -q langgraph langchain-openai langchain-core

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import operator

# LangGraph bruker Ollama via OpenAI-kompatibelt API
llm = ChatOpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
    model="llama3.2",
    temperature=0.1,
)

# Verktøy definert med @tool-dekorator
@tool
def slå_opp_pensjon(tema: str) -> str:
    """Slå opp informasjon om et pensjonema (AFP, alderspensjon, uførepensjon)."""
    database = {
        "afp":           "AFP kan tas fra 62 år med minst 3 års offentlig tjeneste.",
        "alderspensjon": "Alderspensjon fra SPK utbetales fra 67 år, full sats krever 30 år.",
        "uførepensjon":  "Uførepensjon gis ved minst 20% varig nedsatt arbeidsevne.",
    }
    return database.get(tema.lower(), f"Ingen info om '{tema}'.")

@tool
def beregn_månedlig_pensjon(årslønn: int, opptjeningsår: int) -> str:
    """Beregn estimert månedlig pensjon gitt årslønn og opptjeningsår."""
    sats = min(0.66, 0.66 * opptjeningsår / 30)
    return f"{int(årslønn * sats / 12):,} kr/mnd (sats {sats:.0%})"

verktøy = [slå_opp_pensjon, beregn_månedlig_pensjon]
llm_med_verktøy = llm.bind_tools(verktøy)
print("Verktøy klar.")

In [ ]:
# Definer grafen — tilstanden er meldingshistorikken
class AgentTilstand(TypedDict):
    meldinger: Annotated[list, operator.add]  # Akkumuleres automatisk

def agent_node(tilstand: AgentTilstand) -> dict:
    """Agentens 'tenke'-steg: kall LLM."""
    svar = llm_med_verktøy.invoke(tilstand["meldinger"])
    return {"meldinger": [svar]}

def skal_fortsette(tilstand: AgentTilstand) -> str:
    """Bestem neste steg: verktøy eller ferdig?"""
    siste = tilstand["meldinger"][-1]
    if hasattr(siste, "tool_calls") and siste.tool_calls:
        return "verktøy"
    return END

# Bygg grafen
graf = StateGraph(AgentTilstand)
graf.add_node("agent",   agent_node)
graf.add_node("verktøy", ToolNode(verktøy))

graf.set_entry_point("agent")
graf.add_conditional_edges("agent", skal_fortsette)
graf.add_edge("verktøy", "agent")  # Etter verktøy → tilbake til agent

agent = graf.compile()
print("Graf kompilert.")

In [ ]:
# Kjør agenten
resultat = agent.invoke({
    "meldinger": [HumanMessage(content=
        "Jeg har jobbet i staten i 28 år og tjener 700 000 kr. Hva slags pensjon kan jeg forvente?"
    )]
})

print(resultat["meldinger"][-1].content)

---

## Del 2: CrewAI — Team av spesialiserte agenter

In [ ]:
%pip install -q crewai crewai-tools

In [ ]:
from crewai import Agent, Task, Crew, LLM

# CrewAI bruker Ollama via openai/-kompatibelt grensesnitt
ollama_llm = LLM(
    model="ollama/llama3.2",
    base_url="http://localhost:11434",
)

# Agent 1: Pensjonsspesialist
pensjonsspesialist = Agent(
    role="Pensjonsspesialist",
    goal="Gi presis informasjon om pensjonsregler i Norge",
    backstory="Du er ekspert på norsk pensjonssystem med 20 års erfaring hos SPK.",
    llm=ollama_llm,
    verbose=False,
)

# Agent 2: Pensjonskalkulator
kalkulator = Agent(
    role="Pensjonskalkulator",
    goal="Beregne pensjon og gi konkrete tall",
    backstory="Du er aktuarspesialist som beregner pensjoner nøyaktig.",
    llm=ollama_llm,
    verbose=False,
)

print("Agenter klar.")

In [ ]:
# Definer oppgaver
bruker_info = "Alder: 60, tjeneste: 32 år, lønn: 620 000 kr, offentlig ansatt"

oppgave_regler = Task(
    description=f"Forklar hvilke pensjonstyper denne personen har rett til: {bruker_info}",
    expected_output="Liste over pensjonstyper med kvalifikasjonskrav",
    agent=pensjonsspesialist,
)

oppgave_beregning = Task(
    description=f"Beregn estimert månedlig pensjon for: {bruker_info}",
    expected_output="Konkrete månedlige beløp for hver pensjonstype",
    agent=kalkulator,
    context=[oppgave_regler],  # Avhenger av forrige oppgave
)

# Sett opp crew og kjør
crew = Crew(
    agents=[pensjonsspesialist, kalkulator],
    tasks=[oppgave_regler, oppgave_beregning],
    verbose=False,
)

resultat = crew.kickoff()
print(resultat.raw)

---

## Når bruker du hva?

```
Enkel oppgave + én agent   → Manuell ReAct (notatbok 10)
Kompleks flyt + betingelser → LangGraph
Flere spesialiserte roller  → CrewAI
Produksjon + type-sikkerhet → Pydantic AI
```

SPK-annonsen nevner **MCP-servere** — det er neste notatbok.

---

## Hva er neste steg?

**Neste: `12_mcp_and_a2a.ipynb`** — Model Context Protocol: standardisert måte å gi agenter tilgang til verktøy og data. Bygger en enkel MCP-server og kobler den til en agent.